In [169]:
import pandas as pd
import unicodedata
import re
from sklearn.metrics import accuracy_score, classification_report


In [170]:
valid_keywords = [
    'train', 'tgv', 'ter', 'intercités', 'ouigo',
    'aller', 'partir', 'rendre', 'rejoindre', 'voyager',
    'prendre', 'arriver', 'descendre', 'billet', 'réserver',
    'reservation', 'acheter', 'place', 'aller-retour', 'retour',
    'aller simple', 'trajet', 'départ', 'arrivée', 'destination', 'deplacer', 'deplacer de',
    'origine', 'direction', 'vers', 'depuis', 'pour',
    'horaire', 'heure', 'quand', 'prochain', 'disponible',
    'cherche', 'souhaite', 'veux', 'voudrais', 'besoin', 'dois',
    'planning', 'trainée', 'horaires', 'billets', 'réservation'
]

In [171]:
val_df = pd.read_csv('../data/processed/validation_dataset.csv')
ville_df = pd.read_csv('../data/raw/french_town_start.csv')

In [172]:
### Classificateur simple basé sur la présence de mots-clés ###
def normalize(sentence):
    sentence = sentence.lower()
    sentence = unicodedata.normalize('NFKD', sentence).encode('ASCII', 'ignore').decode('utf-8')
    sentence = re.sub(r'[^a-z0-9\s-]', '', sentence)  # retirer ponctuation
    sentence = re.sub(r'\s+', ' ', sentence).strip()    # enlever double espaces
    return sentence

def count_valid_keywords(sentence, threshold=1):
    count = sum(1 for keyword in valid_keywords if keyword in sentence.split())
    return 'VALID' if count >= threshold else 'INVALID'

val_df = pd.read_csv('../data/processed/validation_dataset.csv')
val_df['sentence'] = val_df['sentence'].apply(normalize)
val_df['prediction'] = val_df['sentence'].apply(lambda x: count_valid_keywords(x, threshold=1))

In [173]:
accuracy = accuracy_score(val_df['label'], val_df['prediction'])
print(f"Accuracy sur VALIDATION : {accuracy*100:.2f}%")
print(classification_report(val_df['label'], val_df['prediction']))

Accuracy sur VALIDATION : 94.90%
              precision    recall  f1-score   support

     INVALID       0.95      0.95      0.95       214
       VALID       0.95      0.94      0.95       198

    accuracy                           0.95       412
   macro avg       0.95      0.95      0.95       412
weighted avg       0.95      0.95      0.95       412



In [174]:
towns_df = pd.read_csv('../data/raw/french_town_start.csv')
towns_list = towns_df['nom_ville'].tolist()

In [175]:
### Extracteur simple d'informations ###
def extract_towns(sentence, towns_list):

    sentence_lower = sentence.lower()
    origine_keywords = ['de ', 'depuis ', 'au départ de ', 'en partant de ', 'départ ']
    destination_keywords = ['à ', 'vers ', 'pour ', 'direction ', "jusqu'à ", 'arrivée ']

    origine = None
    destination = None

    towns_with_context = []

    for town in towns_list:
        town_lower = town.lower()
        if town_lower in sentence_lower:
            position = sentence_lower.find(town_lower)

            # Vérifier le contexte avant la ville
            context_start = sentence_lower[max(0,position -20):position]

            is_departure = any(kw in context_start for kw in origine_keywords)
            is_arrival = any(kw in context_start for kw in destination_keywords)

            towns_with_context.append({
                'town': town,
                'position': position,
                'is_departure': is_departure,
                'is_arrival': is_arrival
            })

        for v in towns_with_context:
            if v['is_departure'] and (origine is None or v['position'] < sentence_lower.find(origine.lower())):
                origine = v['town']
            if v['is_arrival'] and (destination is None or v['position'] < sentence_lower.find(destination.lower())):
                destination = v['town']

        if origine is not None and destination is not None:
            towns_ordered = sorted(towns_with_context, key=lambda x: x['position'])
            if len(towns_ordered) > 2:
                if origine is None:
                    origine = towns_ordered[0]['town']
                if destination is None:
                    destination = towns_ordered[1]['town']

    return origine, destination

val_df['extracted_departure'], val_df['extracted_arrival'] = zip(*val_df['sentence'].apply(lambda x: extract_towns(x, towns_list)))

def exact_match(row):
        if pd.isna(row['pred_departure']) or pd.isna(row['pred_arrival']):
            return False
        # Normaliser pour comparer (en minuscules)
        return (str(row['departure']).lower() == str(row['pred_departure']).lower() and
            str(row['arrival']).lower() == str(row['pred_arrival']).lower())

valid_only['correct'] = valid_only.apply(exact_match, axis=1)
extraction_accuracy = valid_only['correct'].mean()


In [177]:
print(f"\n{'='*50}")
print(f"EXTRACTION sur VALIDATION")
print(f"{'='*50}")
print(f"Extraction accuracy : {extraction_accuracy*100:.2f}%")
print(f"Phrases correctement extraites : {valid_only['correct'].sum()}/{len(valid_only)}")

erreurs = valid_only[valid_only['correct'] == False]
print(f"\nNombre d'erreurs : {len(erreurs)}")

if len(erreurs) > 0:
    print(f"\n📋 10 exemples d'erreurs :")
    display(erreurs[['sentence', 'departure', 'arrival', 'pred_departure', 'pred_arrival']].head(10))



EXTRACTION sur VALIDATION
Extraction accuracy : 63.64%
Phrases correctement extraites : 126/198

Nombre d'erreurs : 72

📋 10 exemples d'erreurs :


,sentence,departure,arrival,pred_departure,pred_arrival
24,Train de Rouen à Boulogne-Billancourt cet aprè...,Rouen,Boulogne-Billancourt,Boulogne-Billancourt,Boulogne-Billancourt
29,Train matinal de Cherbourg vers Beauvais,Cherbourg,Beauvais,Beauvais,Beauvais
30,Un billet de train pour aller de Lunéville à C...,Lunéville,Chambéry,Chambéry,Chambéry
35,Je pars en train de Oyonnax vers Saint-Étienne,Oyonnax,Saint-Étienne,Saint-Étienne,Saint-Étienne
37,Je réserve n train de Amboise à La Rochelle,Amboise,La Rochelle,La Rochelle,La Rochelle
41,Train matinal de Dunkerque vers Créteil,Dunkerque,Créteil,Créteil,Créteil
42,Un billet pour le train Avranches Tourcoing,Avranches,Tourcoing,Avranches,Avranches
44,Je planifie un voyage en train de Bastia vers ...,Bastia,Épinal,Épinal,Épinal
46,Je veux partir en milieu de soirée de Montargi...,Montargis,Angoulême,Angoulême,Angoulême
47,TTrain réginoal de Concareau à Aubeas,Concarneau,Aubenas,NaN,NaN
